### Background

This Jupyter Notebook demonstrates how we generated additional data to expand our training set. The main goal of data generation was to address class imbalance and enhance the diversity of our dataset, ultimately improving the performance of our model. We used the OpenAI API as the model provider for data generation.

Before start, you need to install the following dependencies:

```
python-dotenv
openai
pandas
numpy
```

Commanda if you use conda environment:

```
conda activate <env-name>
conda install pandas numpy -y
pip install python-dotenv
pip install openai
```

Create `.env` file with `OPENAI_API_KEY`

### Imports

In [15]:
import os

import numpy as np
import pandas as pd

from dotenv import load_dotenv

from openai import OpenAI

### Init OpenAI client

In [16]:
load_dotenv()

True

In [17]:
client = OpenAI()

### Constants

In [18]:
TRAIN_PATH = "data/"
TRAIN_NAME = "train.parquet"

### Read Data

In [19]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.shape

(3822, 6)

In [20]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [22]:
df = df.loc[~df["techniques"].isna()]
df.shape

(2589, 6)

### Helpers

In [27]:
def extract_phrases(content, trigger_words):
    """
    Extracts phrases from the content based on trigger word indices.

    :param content: The input text.
    :param trigger_words: List of index ranges [[start1, end1], [start2, end2], ...].
    :return: List of extracted phrases.
    """
    return [content[start:end] for start, end in trigger_words]

In [39]:
def check_spans_in_text(text: str, spans: list[str]) -> dict:
    """Check if each span in the list is a substring of the given text."""
    return {span: span in text for span in spans}

### Prompting

In [25]:
def generate_prompt(text: str, triggers: list[str]):
    prompt = f"""You are telegram editor. You need to create a unique samples based on the given text.

Rephrase or generate a similar text to the given text [TEXT].
Leave the phrases defined in list [UNCHANGED] exactly the same (do not highlight them please).

Be creative so the text is not very similar to the given one.
Preserve the meaning and the style of the text. Also, make sure you are using the same language and style as the original text.

[TEXT]
{text}

[UNCHANGED]
{triggers}"""
    return prompt

In [ ]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [32]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [30]:
example = df.sample(1)

techniques = example["techniques"].iloc[0]
text = example["content"].iloc[0]
triggers = extract_phrases(text, example["trigger_words"].iloc[0])
prompt = generate_prompt(text, triggers)

print(techniques)
print("-" * 50)
print(text)
print("-" * 50)
print(triggers)
print("-" * 50)
print(prompt)

['loaded_language' 'fud']
--------------------------------------------------
Офіс Президента Єрмака продовжив рейдерство бізнесу, застосовуючи БЕБ, яке підконтрольне Татарову. 
Тепер атака пішла на Уклон, WOG та ОККО через анонімні канали. 
Бариги, які живуть суто за корупційні гроші, кричать що якісь українські компанії не сплачують в повній мірі податки (законно їх зменшуючи). 
Маніпуляції розраховані на імбецилів. 
А як там 95 квартал, який досі заробляє в Росії транслюючи свої серіали на російських онлайн кінотеатрах, включаючи російський ДЕРЖАВНИЙ Яндекс. 
Про це я вже писав 
тут.
 
Останні рядки на скріні про заробляння на війні - в точку. Як раз цим Зеленський і займається. 
@NovynyPravdy
--------------------------------------------------
['Офіс Президента Єрмака продовжив рейдерство бізнесу, застосовуючи БЕБ, яке підконтрольне Татарову.', 'Бариги, які живуть суто за корупційні гроші, кричать що якісь українські компанії не сплачують в повній мірі податки (законно їх зменшуючи).

In [35]:
%%time

generated_sample = generate_sample(prompt)
print(generated_sample)
print()

Офіс Президента Єрмака продовжив рейдерство бізнесу, застосовуючи БЕБ, яке підконтрольне Татарову. 
Тепер обстріл спрямовано на Уклон, WOG та ОККО через таємні канали.
Бариги, які живуть суто за корупційні гроші, кричать що якісь українські компанії не сплачують в повній мірі податки (законно їх зменшуючи).
Маніпуляції розраховані на імбецилів.
А як поживає 95 квартал, який продовжує отримувати прибуток з Росії, транслюючи свої серіали на російських онлайн-платформах, включаючи державний Яндекс. 
Про це я вже писав у попередніх постах.

Останні рядки на скріні про заробляння на війні - в точку. Як раз цим Зеленський і займається.
@NovynyPravdy

CPU times: user 15.4 ms, sys: 4.32 ms, total: 19.7 ms
Wall time: 5.24 s


In [41]:
result = check_spans_in_text(generated_sample, triggers)
print(result)

{'Офіс Президента Єрмака продовжив рейдерство бізнесу, застосовуючи БЕБ, яке підконтрольне Татарову.': True, 'Бариги, які живуть суто за корупційні гроші, кричать що якісь українські компанії не сплачують в повній мірі податки (законно їх зменшуючи).': True, 'Маніпуляції розраховані на імбецилів.': True, 'Останні рядки на скріні про заробляння на війні - в точку. Як раз цим Зеленський і займається.': True}
